In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from sklearn.manifold import TSNE
from itertools import combinations
import urllib.request, zipfile, os, random, time
import scipy.sparse as sp

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

def set_seed(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

set_seed(42)
os.makedirs('figures', exist_ok=True)

Using device: cpu


In [8]:
import os
import urllib.request
import zipfile

import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
from itertools import combinations

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

os.makedirs('data/movielens', exist_ok=True)
url = 'https://files.grouplens.org/datasets/movielens/ml-100k.zip'
if not os.path.exists('data/movielens/ml-100k/u.data'):
    print('Downloading MovieLens-100k...')
    urllib.request.urlretrieve(url, 'data/movielens/ml-100k.zip')
    with zipfile.ZipFile('data/movielens/ml-100k.zip') as z:
        z.extractall('data/movielens/')
    print('Done!')

ratings = pd.read_csv('data/movielens/ml-100k/u.data', sep='\t',
                       names=['user', 'item', 'rating', 'timestamp'])

genre_cols = ['unknown','Action','Adventure','Animation',"Children's",
              'Comedy','Crime','Documentary','Drama','Fantasy',
              'Film-Noir','Horror','Musical','Mystery','Romance',
              'Sci-Fi','Thriller','War','Western']

movies = pd.read_csv('data/movielens/ml-100k/u.item', sep='|', encoding='latin-1',
                      header=None,
                      names=['item','title','release_date','video_date','imdb_url'] + genre_cols)

movies['year'] = movies['release_date'].str.extract(r'(\d{4})').astype(float)
movies['year'] = movies['year'].fillna(movies['year'].median())
movies['year_norm'] = (movies['year'] - movies['year'].min()) / (movies['year'].max() - movies['year'].min())

feat_cols = genre_cols + ['year_norm']
movie_features = movies[feat_cols].fillna(0).values.astype(np.float32)

movie_ids = sorted(movies.item.unique())
mid2idx = {m: i for i, m in enumerate(movie_ids)}
N_MOVIES = len(movie_ids)

movies_indexed = movies.set_index('item').loc[movie_ids]
genre_matrix = movies_indexed[genre_cols].values
labels = np.array([np.argmax(row) if row.sum() > 0 else 0 for row in genre_matrix])
n_classes = len(set(labels))
print(f'Movies: {N_MOVIES} | Feature dim: {movie_features.shape[1]} | Genre classes: {n_classes}')

# Build co-rating graph: edge if >= MIN_COMMON users rated both movies
MIN_COMMON = 5
user_movies = ratings.groupby('user')['item'].apply(list)
rows, cols = [], []
for user, items in user_movies.items():
    valid = [mid2idx[m] for m in items if m in mid2idx]
    for a, b in combinations(valid, 2):
        rows.append(a); cols.append(b)
        rows.append(b); cols.append(a)

edge_df = pd.DataFrame({'row': rows, 'col': cols})
edge_counts = edge_df.groupby(['row', 'col']).size().reset_index(name='count')
strong_edges = edge_counts[edge_counts['count'] >= MIN_COMMON]
print(f'Edges (co-rating >= {MIN_COMMON}): {len(strong_edges):,}')

A_coo = sp.coo_matrix((np.ones(len(strong_edges)),
                        (strong_edges['row'].values, strong_edges['col'].values)),
                       shape=(N_MOVIES, N_MOVIES))
A_movie = torch.FloatTensor(A_coo.toarray()).to(device)
X_movie = torch.FloatTensor(movie_features).to(device)
Y_movie = torch.LongTensor(labels).to(device)

# Train/val/test split: 20 per class for train
train_mask = torch.zeros(N_MOVIES, dtype=torch.bool)
val_mask   = torch.zeros(N_MOVIES, dtype=torch.bool)
test_mask  = torch.zeros(N_MOVIES, dtype=torch.bool)

for c in range(n_classes):
    idx = (Y_movie == c).nonzero(as_tuple=True)[0]
    if len(idx) >= 20:
        train_mask[idx[:20]] = True
    elif len(idx) > 0:
        train_mask[idx[:len(idx)//2]] = True

remaining = (~train_mask).nonzero(as_tuple=True)[0]
n_val = min(200, len(remaining) // 2)
val_mask[remaining[:n_val]] = True
test_mask[remaining[n_val:n_val+500]] = True

train_mask, val_mask, test_mask = train_mask.to(device), val_mask.to(device), test_mask.to(device)
print(f'Split — Train: {train_mask.sum().item()} | Val: {val_mask.sum().item()} | Test: {test_mask.sum().item()}')

Movies: 1682 | Feature dim: 20 | Genre classes: 19
Edges (co-rating >= 5): 839,578
Split — Train: 267 | Val: 200 | Test: 500


Building blocks (all models + shared training harness)

In [9]:
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

def normalize_adjacency(A):
    """Symmetric normalization: D^{-1/2}(A+I)D^{-1/2}"""
    A_tilde = A + torch.eye(A.size(0), device=A.device)
    D = A_tilde.sum(dim=1)
    D_inv_sqrt = torch.diag(D.pow(-0.5))
    return D_inv_sqrt @ A_tilde @ D_inv_sqrt

# ── GCN (variable depth, for Ex1) ─────────────────────────────────────────
class GCNLayer(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.W = nn.Linear(in_features, out_features, bias=False)
        nn.init.xavier_uniform_(self.W.weight)

    def forward(self, H, A_norm):
        return A_norm @ self.W(H)


class GCNFlexible(nn.Module):
    def __init__(self, in_features, hidden_dim, n_classes, n_layers=2, dropout=0.5):
        super().__init__()
        assert n_layers >= 1
        self.layers = nn.ModuleList()
        if n_layers == 1:
            self.layers.append(GCNLayer(in_features, n_classes))
        else:
            self.layers.append(GCNLayer(in_features, hidden_dim))
            for _ in range(n_layers - 2):
                self.layers.append(GCNLayer(hidden_dim, hidden_dim))
            self.layers.append(GCNLayer(hidden_dim, n_classes))
        self.dropout = nn.Dropout(dropout)

    def forward(self, X, A_norm):
        h = X
        for i, layer in enumerate(self.layers):
            if i == len(self.layers) - 1:
                hidden = h
                logits = layer(h, A_norm)
                return logits, hidden
            h = F.relu(layer(h, A_norm))
            h = self.dropout(h)


# ── GAT ─────────────────────────────────────────────────────────────────
class GATLayer(nn.Module):
    def __init__(self, in_features, out_features, dropout=0.6, alpha=0.2):
        super().__init__()
        self.W = nn.Linear(in_features, out_features, bias=False)
        self.a = nn.Linear(2 * out_features, 1, bias=False)
        self.leaky_relu = nn.LeakyReLU(alpha)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.W.weight)
        nn.init.xavier_uniform_(self.a.weight)

    def forward(self, H, A):
        N = H.size(0)
        Wh = self.W(H)
        Wh_i = Wh.unsqueeze(1).expand(-1, N, -1)
        Wh_j = Wh.unsqueeze(0).expand(N, -1, -1)
        e = self.leaky_relu(self.a(torch.cat([Wh_i, Wh_j], dim=-1)).squeeze(-1))
        mask = (A == 0) & (~torch.eye(N, dtype=torch.bool, device=A.device))
        e = e.masked_fill(mask, float('-inf'))
        alpha = self.dropout(F.softmax(e, dim=1))
        return alpha @ Wh, alpha


class GAT(nn.Module):
    def __init__(self, in_features, hidden_dim, n_classes, n_heads=8, dropout=0.6):
        super().__init__()
        self.heads = nn.ModuleList([GATLayer(in_features, hidden_dim, dropout) for _ in range(n_heads)])
        self.out_layer = GATLayer(hidden_dim * n_heads, n_classes, dropout, alpha=0.2)
        self.dropout = nn.Dropout(dropout)
        self.last_attn = None

    def forward(self, X, A):
        X = self.dropout(X)
        head_outs = [F.elu(head(X, A)[0]) for head in self.heads]
        h = torch.cat(head_outs, dim=-1)
        h = self.dropout(h)
        out, attn = self.out_layer(h, A)
        self.last_attn = attn.detach()
        return out, h


# ── GraphSAGE (mean aggregator, fixed k=10 sampling) ──────────────────────
def sample_neighbors(A, k=10, seed=42):
    rng = np.random.RandomState(seed)
    A_np = A.cpu().numpy()
    N = A_np.shape[0]
    sampled = np.zeros((N, k), dtype=np.int64)
    for i in range(N):
        neighbors = np.nonzero(A_np[i])[0]
        if len(neighbors) == 0:
            sampled[i] = i
        else:
            sampled[i] = rng.choice(neighbors, size=k, replace=len(neighbors) < k)
    return torch.LongTensor(sampled)


class SAGELayer(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.W = nn.Linear(in_features * 2, out_features, bias=True)
        nn.init.xavier_uniform_(self.W.weight)

    def forward(self, H, sampled_idx):
        neighbor_mean = H[sampled_idx].mean(dim=1)
        return self.W(torch.cat([H, neighbor_mean], dim=-1))


class GraphSAGE(nn.Module):
    def __init__(self, in_features, hidden_dim, n_classes, dropout=0.5):
        super().__init__()
        self.layer1 = SAGELayer(in_features, hidden_dim)
        self.layer2 = SAGELayer(hidden_dim, n_classes)
        self.dropout = nn.Dropout(dropout)

    def forward(self, X, sampled_idx):
        h = F.relu(self.layer1(X, sampled_idx))
        h = self.dropout(h)
        return self.layer2(h, sampled_idx), h


# ── MLP baseline (Ex3) ─────────────────────────────────────────────────────
class MLP(nn.Module):
    def __init__(self, in_features, hidden, n_classes, dropout=0.5):
        super().__init__()
        self.fc1 = nn.Linear(in_features, hidden)
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden, n_classes)

    def forward(self, X, A=None):
        h = self.dropout(F.relu(self.fc1(X)))
        return self.fc2(h), h


# ── Shared training harness ────────────────────────────────────────────────
def train_node_classifier(model, X, A, Y, train_mask, val_mask, test_mask,
                           epochs=200, lr=0.01, weight_decay=5e-4):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    t0 = time.time()
    for epoch in range(epochs):
        model.train()
        logits, _ = model(X, A)
        loss = F.cross_entropy(logits[train_mask], Y[train_mask])
        optimizer.zero_grad(); loss.backward(); optimizer.step()
    avg_epoch_time = (time.time() - t0) / epochs

    model.eval()
    with torch.no_grad():
        logits, embeddings = model(X, A)
        test_acc = (logits[test_mask].argmax(1) == Y[test_mask]).float().mean().item()
    return test_acc, embeddings, avg_epoch_time


def avg_pairwise_cosine_sim(embeddings, mask):
    emb = F.normalize(embeddings[mask], dim=1)
    sim = emb @ emb.T
    n = sim.size(0)
    return ((sim.sum() - sim.diagonal().sum()) / (n * (n - 1))).item()


# ── Precompute shared structures ───────────────────────────────────────────
A_norm = normalize_adjacency(A_movie)
sampled_idx_10 = sample_neighbors(A_movie, k=10).to(device)
print('Building blocks ready.')

Building blocks ready.


Cell 3 — Building blocks (all models + shared training harness)

In [10]:
def normalize_adjacency(A):
    """Symmetric normalization: D^{-1/2}(A+I)D^{-1/2}"""
    A_tilde = A + torch.eye(A.size(0), device=A.device)
    D = A_tilde.sum(dim=1)
    D_inv_sqrt = torch.diag(D.pow(-0.5))
    return D_inv_sqrt @ A_tilde @ D_inv_sqrt

# ── GCN (variable depth, for Ex1) ─────────────────────────────────────────
class GCNLayer(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.W = nn.Linear(in_features, out_features, bias=False)
        nn.init.xavier_uniform_(self.W.weight)

    def forward(self, H, A_norm):
        return A_norm @ self.W(H)


class GCNFlexible(nn.Module):
    def __init__(self, in_features, hidden_dim, n_classes, n_layers=2, dropout=0.5):
        super().__init__()
        assert n_layers >= 1
        self.layers = nn.ModuleList()
        if n_layers == 1:
            self.layers.append(GCNLayer(in_features, n_classes))
        else:
            self.layers.append(GCNLayer(in_features, hidden_dim))
            for _ in range(n_layers - 2):
                self.layers.append(GCNLayer(hidden_dim, hidden_dim))
            self.layers.append(GCNLayer(hidden_dim, n_classes))
        self.dropout = nn.Dropout(dropout)

    def forward(self, X, A_norm):
        h = X
        for i, layer in enumerate(self.layers):
            if i == len(self.layers) - 1:
                hidden = h
                logits = layer(h, A_norm)
                return logits, hidden
            h = F.relu(layer(h, A_norm))
            h = self.dropout(h)


# ── GAT ─────────────────────────────────────────────────────────────────
class GATLayer(nn.Module):
    def __init__(self, in_features, out_features, dropout=0.6, alpha=0.2):
        super().__init__()
        self.W = nn.Linear(in_features, out_features, bias=False)
        self.a = nn.Linear(2 * out_features, 1, bias=False)
        self.leaky_relu = nn.LeakyReLU(alpha)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.W.weight)
        nn.init.xavier_uniform_(self.a.weight)

    def forward(self, H, A):
        N = H.size(0)
        Wh = self.W(H)
        Wh_i = Wh.unsqueeze(1).expand(-1, N, -1)
        Wh_j = Wh.unsqueeze(0).expand(N, -1, -1)
        e = self.leaky_relu(self.a(torch.cat([Wh_i, Wh_j], dim=-1)).squeeze(-1))
        mask = (A == 0) & (~torch.eye(N, dtype=torch.bool, device=A.device))
        e = e.masked_fill(mask, float('-inf'))
        alpha = self.dropout(F.softmax(e, dim=1))
        return alpha @ Wh, alpha


class GAT(nn.Module):
    def __init__(self, in_features, hidden_dim, n_classes, n_heads=8, dropout=0.6):
        super().__init__()
        self.heads = nn.ModuleList([GATLayer(in_features, hidden_dim, dropout) for _ in range(n_heads)])
        self.out_layer = GATLayer(hidden_dim * n_heads, n_classes, dropout, alpha=0.2)
        self.dropout = nn.Dropout(dropout)
        self.last_attn = None

    def forward(self, X, A):
        X = self.dropout(X)
        head_outs = [F.elu(head(X, A)[0]) for head in self.heads]
        h = torch.cat(head_outs, dim=-1)
        h = self.dropout(h)
        out, attn = self.out_layer(h, A)
        self.last_attn = attn.detach()
        return out, h


# ── GraphSAGE (mean aggregator, fixed k=10 sampling) ──────────────────────
def sample_neighbors(A, k=10, seed=42):
    rng = np.random.RandomState(seed)
    A_np = A.cpu().numpy()
    N = A_np.shape[0]
    sampled = np.zeros((N, k), dtype=np.int64)
    for i in range(N):
        neighbors = np.nonzero(A_np[i])[0]
        if len(neighbors) == 0:
            sampled[i] = i
        else:
            sampled[i] = rng.choice(neighbors, size=k, replace=len(neighbors) < k)
    return torch.LongTensor(sampled)


class SAGELayer(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.W = nn.Linear(in_features * 2, out_features, bias=True)
        nn.init.xavier_uniform_(self.W.weight)

    def forward(self, H, sampled_idx):
        neighbor_mean = H[sampled_idx].mean(dim=1)
        return self.W(torch.cat([H, neighbor_mean], dim=-1))


class GraphSAGE(nn.Module):
    def __init__(self, in_features, hidden_dim, n_classes, dropout=0.5):
        super().__init__()
        self.layer1 = SAGELayer(in_features, hidden_dim)
        self.layer2 = SAGELayer(hidden_dim, n_classes)
        self.dropout = nn.Dropout(dropout)

    def forward(self, X, sampled_idx):
        h = F.relu(self.layer1(X, sampled_idx))
        h = self.dropout(h)
        return self.layer2(h, sampled_idx), h


# ── MLP baseline (Ex3) ─────────────────────────────────────────────────────
class MLP(nn.Module):
    def __init__(self, in_features, hidden, n_classes, dropout=0.5):
        super().__init__()
        self.fc1 = nn.Linear(in_features, hidden)
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden, n_classes)

    def forward(self, X, A=None):
        h = self.dropout(F.relu(self.fc1(X)))
        return self.fc2(h), h


# ── Shared training harness ────────────────────────────────────────────────
def train_node_classifier(model, X, A, Y, train_mask, val_mask, test_mask,
                           epochs=200, lr=0.01, weight_decay=5e-4):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    t0 = time.time()
    for epoch in range(epochs):
        model.train()
        logits, _ = model(X, A)
        loss = F.cross_entropy(logits[train_mask], Y[train_mask])
        optimizer.zero_grad(); loss.backward(); optimizer.step()
    avg_epoch_time = (time.time() - t0) / epochs

    model.eval()
    with torch.no_grad():
        logits, embeddings = model(X, A)
        test_acc = (logits[test_mask].argmax(1) == Y[test_mask]).float().mean().item()
    return test_acc, embeddings, avg_epoch_time


def avg_pairwise_cosine_sim(embeddings, mask):
    emb = F.normalize(embeddings[mask], dim=1)
    sim = emb @ emb.T
    n = sim.size(0)
    return ((sim.sum() - sim.diagonal().sum()) / (n * (n - 1))).item()


# ── Precompute shared structures ───────────────────────────────────────────
A_norm = normalize_adjacency(A_movie)
sampled_idx_10 = sample_neighbors(A_movie, k=10).to(device)
print('Building blocks ready.')

Building blocks ready.


 Ex1: Over-smoothing (depth sweep)

In [ ]:
depths = [1, 2, 3, 4, 5]
depth_results = {}

for n_layers in depths:
    set_seed(42)
    model = GCNFlexible(X_movie.shape[1], 64, n_classes, n_layers=n_layers).to(device)
    test_acc, embeddings, _ = train_node_classifier(model, X_movie, A_norm, Y_movie,
                                                      train_mask, val_mask, test_mask, epochs=200)
    cos_sim = avg_pairwise_cosine_sim(embeddings, test_mask)
    depth_results[n_layers] = (test_acc, cos_sim)
    print(f'Layers: {n_layers} | Test Acc: {test_acc*100:.2f}% | Avg Cosine Sim: {cos_sim:.4f}')

fig, ax1 = plt.subplots(figsize=(7, 5))
layers_list = list(depth_results.keys())
accs = [depth_results[k][0] for k in layers_list]
sims = [depth_results[k][1] for k in layers_list]

ax1.plot(layers_list, accs, 'o-', color='tab:blue', label='Test Accuracy')
ax1.set_xlabel('Number of GCN Layers'); ax1.set_ylabel('Test Accuracy', color='tab:blue')
ax1.tick_params(axis='y', labelcolor='tab:blue')
ax2 = ax1.twinx()
ax2.plot(layers_list, sims, 's--', color='tab:red', label='Avg Cosine Similarity')
ax2.set_ylabel('Avg Pairwise Cosine Similarity', color='tab:red')
ax2.tick_params(axis='y', labelcolor='tab:red')
plt.title('Over-smoothing: Accuracy & Embedding Similarity vs Depth')
fig.tight_layout()
plt.savefig('figures/over_smoothing.png', dpi=150)
plt.show()

Building blocks ready.


GAT beats GCN by the largest margin on heterophilic/heterogeneous graphs, where a
node's neighbors vary a lot in relevance (e.g. a citation graph mixing closely-related
and tangential citations) — GAT learns to downweight noisy edges, GCN treats them equally.

GraphSAGE beats GCN most on very large or dynamic graphs where full N×N adjacency
operations are infeasible. Its neighbor sampling scales to millions of nodes and lets
it generalize inductively to unseen nodes at test time — something transductive GCN,
which needs the full fixed adjacency, cannot do.

Ex3: MLP baseline + final comparison

In [ ]:
set_seed(42)
mlp = MLP(X_movie.shape[1], 64, n_classes).to(device)
acc_mlp, emb_mlp, _ = train_node_classifier(mlp, X_movie, None, Y_movie,
                                             train_mask, val_mask, test_mask, epochs=200)

print(f"{'Model':<18}{'Test Accuracy':>15}")
print(f"{'MLP (no graph)':<18}{acc_mlp*100:>14.2f}%")
print(f"{'GCN':<18}{acc_gcn*100:>14.2f}%")
print(f"{'GAT':<18}{acc_gat*100:>14.2f}%")
print(f"{'GraphSAGE':<18}{acc_sage*100:>14.2f}%")
print(f"\nGraph structure improves accuracy by {(acc_gcn-acc_mlp)*100:.2f} points over MLP (best graph model: GCN).")

Building blocks ready.


 t-SNE comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
colors_map = plt.cm.tab20(np.linspace(0, 1, n_classes))

for ax, (name, emb) in zip(axes, [('GCN', emb_gcn2), ('GAT', emb_gat), ('GraphSAGE', emb_sage)]):
    proj = TSNE(n_components=2, random_state=42, perplexity=30).fit_transform(emb.cpu().detach().numpy())
    for c in range(n_classes):
        mask = labels_np == c
        if mask.any():
            ax.scatter(proj[mask, 0], proj[mask, 1], c=[colors_map[c]], label=genre_cols[c], alpha=0.7, s=12)
    ax.set_title(f'{name} Embeddings (t-SNE)'); ax.axis('off')
axes[0].legend(fontsize=6, markerscale=2, ncol=2, loc='upper left')
plt.suptitle('Learned Movie Representations — Genre Clustering')
plt.tight_layout()
plt.savefig('figures/tsne_embeddings.png', dpi=150)
plt.show()

 Ex4: Bipartite graph + RecGCN vs LightGCN

In [ ]:
pos_ratings = ratings[ratings.rating >= 4].copy()
user_ids = sorted(pos_ratings.user.unique())
item_ids = sorted(pos_ratings.item.unique())
user2idx = {u: i for i, u in enumerate(user_ids)}
item2idx = {it: i for i, it in enumerate(item_ids)}
N_USERS, N_ITEMS = len(user_ids), len(item_ids)
N_NODES = N_USERS + N_ITEMS
EMBED_DIM = 32

pos_ratings['u_idx'] = pos_ratings.user.map(user2idx)
pos_ratings['i_idx'] = pos_ratings.item.map(item2idx) + N_USERS
all_pos = list(zip(pos_ratings.u_idx, pos_ratings.i_idx))
random.shuffle(all_pos)
split = int(0.8 * len(all_pos))
train_pos, test_pos = all_pos[:split], all_pos[split:]
pos_set = set(all_pos)

def sample_negatives(n, pos_set, n_users, n_items):
    negs = []
    while len(negs) < n:
        u = random.randint(0, n_users - 1)
        i = random.randint(0, n_items - 1) + n_users
        if (u, i) not in pos_set:
            negs.append((u, i))
    return negs

train_neg = sample_negatives(len(train_pos), pos_set, N_USERS, N_ITEMS)
test_neg = sample_negatives(len(test_pos), pos_set, N_USERS, N_ITEMS)

A_rec = torch.zeros(N_NODES, N_NODES)
for u, v in train_pos:
    A_rec[u, v] = 1; A_rec[v, u] = 1
A_rec_norm = normalize_adjacency(A_rec.to(device))
print(f'Users: {N_USERS} | Movies: {N_ITEMS} | Train edges: {len(train_pos):,} | Test edges: {len(test_pos):,}')


class RecGCN(nn.Module):
    """GCN encoder: n_layers of (W, ReLU), same width throughout."""
    def __init__(self, embed_dim=32, n_layers=3):
        super().__init__()
        self.layers = nn.ModuleList([GCNLayer(embed_dim, embed_dim) for _ in range(n_layers)])

    def forward(self, X, A_norm):
        h = X
        for layer in self.layers:
            h = F.relu(layer(h, A_norm))
        return h

    def predict(self, embeddings, u, v):
        return torch.sigmoid((embeddings[u] * embeddings[v]).sum(dim=-1))


class LightGCN(nn.Module):
    """Pure propagation, no W, no activation. Final = mean over layers 0..K."""
    def __init__(self, n_layers=3):
        super().__init__()
        self.n_layers = n_layers

    def forward(self, X, A_norm):
        embs = [X]
        h = X
        for _ in range(self.n_layers):
            h = A_norm @ h
            embs.append(h)
        return torch.stack(embs, dim=0).mean(dim=0)

    def predict(self, embeddings, u, v):
        return torch.sigmoid((embeddings[u] * embeddings[v]).sum(dim=-1))


def train_rec_model(model, user_emb_layer, item_emb_layer, A_norm, train_pos, train_neg, epochs=30, lr=1e-2):
    params = list(model.parameters()) + list(user_emb_layer.parameters()) + list(item_emb_layer.parameters())
    optimizer = torch.optim.Adam(params, lr=lr)
    pos_u = torch.tensor([e[0] for e in train_pos], device=device)
    pos_v = torch.tensor([e[1] for e in train_pos], device=device)
    neg_u = torch.tensor([e[0] for e in train_neg], device=device)
    neg_v = torch.tensor([e[1] for e in train_neg], device=device)
    for epoch in range(epochs):
        model.train()
        node_feat = torch.cat([user_emb_layer.weight, item_emb_layer.weight], dim=0)
        emb = model(node_feat, A_norm)
        pos_scores = model.predict(emb, pos_u, pos_v)
        neg_scores = model.predict(emb, neg_u, neg_v)
        loss = -torch.log(pos_scores + 1e-8).mean() - torch.log(1 - neg_scores + 1e-8).mean()
        optimizer.zero_grad(); loss.backward(); optimizer.step()
    model.eval()
    with torch.no_grad():
        node_feat = torch.cat([user_emb_layer.weight, item_emb_layer.weight], dim=0)
        emb = model(node_feat, A_norm)
    return emb


def evaluate_rec(model, emb, test_pos, test_neg, N_USERS, N_ITEMS, K=10):
    tpu = torch.tensor([e[0] for e in test_pos], device=device)
    tpv = torch.tensor([e[1] for e in test_pos], device=device)
    tnu = torch.tensor([e[0] for e in test_neg], device=device)
    tnv = torch.tensor([e[1] for e in test_neg], device=device)
    with torch.no_grad():
        pos_sc = model.predict(emb, tpu, tpv)
        neg_sc = model.predict(emb, tnu, tnv)
    auc = ((pos_sc.unsqueeze(1) > neg_sc.unsqueeze(0)).float().mean()).item()

    user_test = {}
    for u, v in test_pos:
        user_test.setdefault(u, []).append(v)
    recalls = []
    with torch.no_grad():
        for user_node, true_items in list(user_test.items())[:100]:
            item_nodes = torch.arange(N_USERS, N_USERS + N_ITEMS, device=device)
            u_emb_rep = emb[user_node].unsqueeze(0).expand(N_ITEMS, -1)
            scores = torch.sigmoid((emb[item_nodes] * u_emb_rep).sum(dim=-1))
            top_k = scores.topk(K).indices.cpu().numpy() + N_USERS
            recalls.append(sum(1 for it in true_items if it in top_k) / len(true_items))
    return auc, np.mean(recalls)


set_seed(42)
user_emb_gcn = nn.Embedding(N_USERS, EMBED_DIM).to(device)
item_emb_gcn = nn.Embedding(N_ITEMS, EMBED_DIM).to(device)
rec_gcn = RecGCN(embed_dim=EMBED_DIM, n_layers=3).to(device)
emb_gcn_rec = train_rec_model(rec_gcn, user_emb_gcn, item_emb_gcn, A_rec_norm, train_pos, train_neg)
auc_gcn, recall_gcn = evaluate_rec(rec_gcn, emb_gcn_rec, test_pos, test_neg, N_USERS, N_ITEMS)
params_gcn = sum(p.numel() for p in rec_gcn.parameters()) + sum(p.numel() for p in user_emb_gcn.parameters()) + sum(p.numel() for p in item_emb_gcn.parameters())

set_seed(42)
user_emb_light = nn.Embedding(N_USERS, EMBED_DIM).to(device)
item_emb_light = nn.Embedding(N_ITEMS, EMBED_DIM).to(device)
light_gcn = LightGCN(n_layers=3).to(device)
emb_light = train_rec_model(light_gcn, user_emb_light, item_emb_light, A_rec_norm, train_pos, train_neg)
auc_light, recall_light = evaluate_rec(light_gcn, emb_light, test_pos, test_neg, N_USERS, N_ITEMS)
params_light = sum(p.numel() for p in light_gcn.parameters()) + sum(p.numel() for p in user_emb_light.parameters()) + sum(p.numel() for p in item_emb_light.parameters())

print(f"{'Model':<20}{'#Params':>10}{'AUC':>10}{'Recall@10':>12}")
print(f"{'RecGCN (with W)':<20}{params_gcn:>10,}{auc_gcn:>10.4f}{recall_gcn:>12.4f}")
print(f"{'LightGCN (no W)':<20}{params_light:>10,}{auc_light:>10.4f}{recall_light:>12.4f}")

Ex4c: LightGCN depth sweep

In [ ]:
light_depths = [1, 2, 3, 4]
light_recall_results = {}

for nl in light_depths:
    set_seed(42)
    u_emb = nn.Embedding(N_USERS, EMBED_DIM).to(device)
    i_emb = nn.Embedding(N_ITEMS, EMBED_DIM).to(device)
    model = LightGCN(n_layers=nl).to(device)
    emb = train_rec_model(model, u_emb, i_emb, A_rec_norm, train_pos, train_neg)
    _, recall = evaluate_rec(model, emb, test_pos, test_neg, N_USERS, N_ITEMS)
    light_recall_results[nl] = recall
    print(f'n_layers={nl} | Recall@10: {recall:.4f}')

plt.figure(figsize=(6, 4))
plt.plot(list(light_recall_results.keys()), list(light_recall_results.values()), 'o-', color='tab:green')
plt.xlabel('Number of LightGCN Layers'); plt.ylabel('Recall@10')
plt.title('LightGCN: Recall@10 vs Depth')
plt.tight_layout()
plt.savefig('figures/lightgcn_recall_vs_depth.png', dpi=150)
plt.show()